# Data Cleaning & Feature Engineering

## Project
ABC Bank Customer Churn Analysis

### Purpose
The purpose of this notebook is to prepare the customer dataset for downstream business analysis. The workflow includes data inspection, data quality assessment, data cleaning, and feature engineering to ensure the dataset is suitable for SQL analysis, Tableau visualization, and predictive modeling in R.

### 1. Import Library
Import the Python libraries required for data manipulation and analysis.

In [5]:
import pandas as pd
import numpy as np


### 2. Load Dataset
Load the original customer churn dataset and preview its structure and inspect the dataset to understand its dimensions, data types, and summary statistics before performing any cleaning.


In [6]:
df = pd.read_csv("/Users/hienle/Desktop/Bank customer churn/Bank Customer Churn Prediction-selected-columns.csv")


In [7]:
## first 5 rows
df.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member
0,15634602,619,France,Female,42,2,0.00,1,1,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1
2,15619304,502,France,Female,42,8,159660.80,3,1,0
3,15701354,699,France,Female,39,1,0.00,2,0,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1


In [8]:
## last 5 rows
df.tail()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member
9995,15606229,771,France,Male,39,5,0.00,2,1,0
9996,15569892,516,France,Male,35,10,57369.61,1,1,1
9997,15584532,709,France,Female,36,7,0.00,1,0,1
9998,15682355,772,Germany,Male,42,3,75075.31,2,1,0
9999,15628319,792,France,Female,28,4,130142.79,1,1,0


In [9]:
df.shape


(10000, 10)

In [10]:
df.info

<bound method DataFrame.info of       customer_id  credit_score  country  gender  age  tenure    balance  \
0        15634602           619   France  Female   42       2       0.00   
1        15647311           608    Spain  Female   41       1   83807.86   
2        15619304           502   France  Female   42       8  159660.80   
3        15701354           699   France  Female   39       1       0.00   
4        15737888           850    Spain  Female   43       2  125510.82   
...           ...           ...      ...     ...  ...     ...        ...   
9995     15606229           771   France    Male   39       5       0.00   
9996     15569892           516   France    Male   35      10   57369.61   
9997     15584532           709   France  Female   36       7       0.00   
9998     15682355           772  Germany    Male   42       3   75075.31   
9999     15628319           792   France  Female   28       4  130142.79   

      products_number  credit_card  active_member  
0  

The dataset contains 10,000 customer records and 10 variables.


In [11]:
df.describe()

,customer_id,credit_score,age,tenure,balance,products_number,credit_card,active_member
count,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000
mean,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100
std,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797
min,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000
25%,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000
50%,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000
75%,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000
max,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000


### 3. Check Missing Value
Identify whether the dataset contains any missing values that may affect subsequent analysis.


In [12]:
df.isnull().sum()

customer_id        0
credit_score       0
country            0
gender             0
age                0
tenure             0
balance            0
products_number    0
credit_card        0
active_member      0
dtype: int64

No missing values were detected.

   ### 4. Check duplicate
   Ensure that each customer is represented only once.

   

In [13]:
df.duplicated().sum()

np.int64(0)

### 5. Check unique value

In [14]:
df["age"].describe()

count    10000.000000
mean        38.921800
std         10.487806
min         18.000000
25%         32.000000
50%         37.000000
75%         44.000000
max         92.000000
Name: age, dtype: float64

In [15]:
df['gender'].value_counts()

gender
Male      5457
Female    4543
Name: count, dtype: int64

In [17]:
df['age'].value_counts()

age
37    478
38    477
35    474
36    456
34    447
     ... 
92      2
82      1
88      1
85      1
83      1
Name: count, Length: 70, dtype: int64

### 6. Feature Engineering

1. Create Age Group

Customers are grouped into age categories to simplify business reporting and enable churn comparisons across different age segments.


In [16]:
## Age Group
bins = [18, 30, 40, 50, 100]

labels = [
    "18-30",
    "31-40",
    "41-50",
    "50+"
]

df["age_group"] = pd.cut(
    df["age"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
df.head()


,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,age_group
0,15634602,619,France,Female,42,2,0.00,1,1,1,41-50
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,41-50
2,15619304,502,France,Female,42,8,159660.80,3,1,0,41-50
3,15701354,699,France,Female,39,1,0.00,2,0,0,31-40
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,41-50


2. Ceate Credit Score
    
Credit scores are categorized into standard creditworthiness levels (Poor, Fair, Good, Very Good, and Excellent) to facilitate customer segmentation.


In [17]:
conditions = [
    df["credit_score"] < 580,
    (df["credit_score"] >= 580) & (df["credit_score"] < 670),
    (df["credit_score"] >= 670) & (df["credit_score"] < 740),
    (df["credit_score"] >= 740) & (df["credit_score"] < 800),
    df["credit_score"] >= 800
]

choices = [
    "Poor",
    "Fair",
    "Good",
    "Very Good",
    "Excellent"
]

df["credit_score_group"] = np.select(
    conditions,
    choices,
    default="Unknown"
)
df.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,age_group,credit_score_group
0,15634602,619,France,Female,42,2,0.00,1,1,1,41-50,Fair
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,41-50,Fair
2,15619304,502,France,Female,42,8,159660.80,3,1,0,41-50,Poor
3,15701354,699,France,Female,39,1,0.00,2,0,0,31-40,Good
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,41-50,Excellent


3. Create Balance Group

Approximately 36% of customers have a zero account balance. Therefore, customers with zero balances are treated as a separate segment. Positive balances are then classified into Low, Medium, and High groups based on the observed data distribution.


In [18]:
(df["balance"] == 0).sum()

np.int64(3617)

In [19]:
df[df["balance"] > 0]["balance"].describe()

count      6383.000000
mean     119827.493793
std       30095.056462
min        3768.690000
25%      100181.975000
50%      119839.690000
75%      139512.290000
max      250898.090000
Name: balance, dtype: float64

In [20]:
def balance_group(balance):
    if balance == 0:
        return "Zero Balance"
    elif balance <= 100000:
        return "Low Balance"
    elif balance <= 140000:
        return "Medium Balance"
    else:
        return "High Balance"

df["balance_group"] = df["balance"].apply(balance_group)
df.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,age_group,credit_score_group,balance_group
0,15634602,619,France,Female,42,2,0.00,1,1,1,41-50,Fair,Zero Balance
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,41-50,Fair,Low Balance
2,15619304,502,France,Female,42,8,159660.80,3,1,0,41-50,Poor,High Balance
3,15701354,699,France,Female,39,1,0.00,2,0,0,31-40,Good,Zero Balance
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,41-50,Excellent,Medium Balance


### 7. Export Clean Dataset
Save the cleaned dataset for downstream SQL analysis, Tableau dashboards, and predictive modeling in R.


In [22]:
df.to_csv(
    "/Users/hienle/Desktop/bank_customer_churn_cleaned.csv",
    index=False
)

In [4]:
import sqlite3
import pandas as pd



In [10]:
df = pd.read_csv("/Users/hienle/Desktop/Bank customer churn/Bank Customer Churn Prediction-selected-columns.csv")
conn = sqlite3.connect("bank_churn.db")

df.to_sql("bank_churn", conn, if_exists="replace", index=False)
conn.close()
